 # Fundamentals — "Implement X from scratch"

## ReLU

ReLU = max(0,x)

In [1]:
import torch

In [2]:
def relu(x:torch.Tensor) -> torch.Tensor:
  return x * (x>0).float()

In [3]:
x = torch.randn(3,3)

In [4]:
relu(x)

tensor([[0.0127, 1.1944, 0.6702],
        [-0.0000, 0.2966, -0.0000],
        [0.5027, 0.9169, -0.0000]])

## Softmax
$$\text{softmax}(x_i) = \frac{e^{x_i-max(x)}}{\sum_j e^{x_j-max(x)}}$$

In [5]:
def softmax(x:torch.Tensor, dim: int = -1) -> torch.Tensor:
  x_max = torch.max(x, dim=dim, keepdim=True).values
  e_x = torch.exp(x-x_max)
  return e_x / torch.sum(e_x, dim=dim, keepdim=True)

In [6]:
x = torch.tensor([-3.0, -2.0, -1.0])

In [7]:
softmax(x)

tensor([0.0900, 0.2447, 0.6652])

##Cross-Entropy Loss
$$CE(x,y) = -log(\frac{e^{x_y}}{\sum_j e^{x_j}})$$

In [8]:
def cross_entropy_loss(logits, target):
  log_probs = logits - torch.logsumexp(logits, dim=1, keepdim=True)
  return -log_probs[torch.arange(target.shape[0]), target].mean()

In [9]:
logits = torch.randn(4,10)
target = torch.randint(0,10,(4,))

In [10]:
cross_entropy_loss(logits, target)

tensor(2.8903)

## Dropout
$$h' = \frac{h.m}{1-p}$$

In [11]:
import torch.nn as nn
class Dropout(nn.Module):
  def __init__(self, p=0.5):
    super().__init__()
    self.p = p
  def forward(self, x):
    if not self.training or self.p == 0:
      return x
    mask = (torch.rand_like(x) > self.p).float()
    return x * mask / (1-self.p)




In [12]:
d = Dropout(p=0.5)

In [13]:
d.train()

Dropout()

In [14]:
x = torch.ones(10)

In [15]:
d(x)

tensor([0., 2., 2., 2., 2., 2., 2., 2., 2., 2.])

In [16]:
d.eval()

Dropout()

In [17]:
d(x)

tensor([1., 1., 1., 1., 1., 1., 1., 1., 1., 1.])

## Embedding


In [18]:
class Embedding(nn.Module):
  def __init__(self, num_embeddings, embedding_dim):
    super().__init__()
    self.weight = nn.Parameter(torch.randn(num_embeddings, embedding_dim))
  def forward(self, indices):
    return self.weight[indices]

In [19]:
emb = Embedding(10,3)
idx = torch.tensor([1,2,3])

In [20]:
emb(idx)

tensor([[ 0.6618, -1.9867,  0.0708],
        [-0.4467,  0.8150, -1.4520],
        [-0.3746, -0.0520, -1.1783]], grad_fn=<IndexBackward0>)

## GeLU

$$gelu(x) = 0.5 . x .(1 + tanh(\sqrt(\frac{2}{\pi})(x+0.044715.x^3)$$

In [21]:
import math

In [22]:
def gelu(x):
  return 0.5*x*(1+torch.erf(x/math.sqrt(2.0)))

In [23]:
x = torch.tensor([-2., -1., 0., 1., 2.])

In [24]:
gelu(x)

tensor([-0.0455, -0.1587,  0.0000,  0.8413,  1.9545])

## Kaiming Init
$$W \sim \mathcal{N}(0, \text{std}^2) \quad \text{where} \quad \text{std} = \sqrt{\frac{2}{\text{fan_in}}}$$

In [25]:
def kaiming_init(weight):
  fan_in = weight.shape[1] if weight.dim()>=2 else weight.shape[0]
  std = math.sqrt(2.0/fan_in)
  with torch.no_grad():
    weight.normal_(0, std)
  return weight


In [26]:
w = torch.empty(256,512)
kaiming_init(w)

tensor([[ 0.0546,  0.0442, -0.0046,  ..., -0.0546, -0.0195,  0.0071],
        [-0.0915, -0.1474, -0.1784,  ...,  0.0442, -0.0108,  0.0324],
        [-0.0532,  0.0594, -0.0746,  ..., -0.0838,  0.0580,  0.0042],
        ...,
        [ 0.0984, -0.1115, -0.0191,  ..., -0.0403, -0.0165,  0.0041],
        [-0.0923,  0.0280, -0.0316,  ..., -0.0797, -0.0385,  0.1090],
        [ 0.0366,  0.0625, -0.0184,  ..., -0.0333, -0.0370, -0.0142]])

In [27]:
w.mean()

tensor(0.0003)

In [28]:
w.std()

tensor(0.0624)

## Gradient Norm Clipping

In [29]:
def clip_grad_norm(parameters, max_norm):
  parameters = [p for p in parameters if p.grad is not None]
  total_norm = torch.sqrt(sum(p.grad.norm()**2 for p in parameters))
  clip_coef = max_norm / (total_norm + 1e-6)
  if clip_coef < 1:
    for p in parameters:
      p.grad.mul_(clip_coef)
  return total_norm.item()

In [30]:
p = torch.randn(100, requires_grad=True)
(p*10).sum().backward()
orig_norm = clip_grad_norm([p], max_norm=1.0)

In [31]:
p

tensor([ 2.5644e+00, -5.8332e-01,  1.4609e+00, -5.0912e-01, -1.2450e+00,
        -3.9171e-01, -1.1433e-01, -2.5084e-01, -1.4705e-01, -7.6340e-01,
        -9.3436e-01, -3.7656e-01, -7.8086e-01, -8.7658e-01, -1.2485e-01,
        -1.0402e+00,  3.6746e-01,  2.6195e-01,  2.3979e-01, -1.9237e-01,
         1.2706e+00, -5.2073e-01,  8.7215e-01, -1.7857e-01,  2.0304e-01,
        -3.5251e-02, -1.2111e+00, -1.1556e+00, -3.8552e-01,  2.1131e+00,
        -6.2871e-01, -1.9356e+00,  2.6650e-01,  1.0290e+00, -3.9994e-01,
        -1.7183e+00, -1.0716e+00, -6.3248e-02,  1.1394e+00,  1.7510e-03,
        -1.6867e+00,  8.5991e-01, -7.2324e-01, -1.5229e-01,  1.3295e+00,
        -8.7226e-01,  2.9084e-01, -2.6985e-01, -5.1629e-01, -1.7956e-01,
         2.5122e-01, -4.7061e-01, -1.9229e+00, -2.2398e+00,  3.9707e-01,
         1.0587e-01, -1.4452e+00,  7.4569e-03, -1.2791e-01, -2.8489e-01,
         2.7451e-01,  4.2570e-01,  1.2072e+00,  1.2789e+00,  9.0073e-01,
        -8.3003e-01, -1.2976e+00,  2.1773e-01, -3.4

In [32]:
orig_norm

100.0

In [33]:
p.grad.norm().item()

0.9999999403953552

## Linear Layer
$$y = X.W^T +b$$

In [34]:
class SimpleLinear:
  def __init__(self, in_features:int, out_features:int):
    self.weight = torch.randn(out_features, in_features, requires_grad=True)
    self.bias = torch.randn(out_features, requires_grad=True)
  def forward(self, x:torch.Tensor) -> torch.Tensor:
    return x @ self.weight.T +self.bias

In [35]:
layer = SimpleLinear(4,2)
x = torch.randn(3,4)
y = layer.forward(x)

In [36]:
x

tensor([[ 3.1008, -1.6935,  0.1866,  1.9631],
        [-0.3958, -0.6978,  0.3017,  0.1487],
        [-1.1336,  1.6691,  2.0680,  1.0466]])

In [37]:
y

tensor([[-2.5622, -0.6597],
        [-3.0886,  0.1936],
        [-3.0376,  0.6280]], grad_fn=<AddBackward0>)

##Gradient Accumulation

In [38]:
def accumulation_step(model, optimizer, loss_fn, micro_batches):
  optimizer.zero_grad()
  total_loss = 0.0
  n = len(micro_batches)
  for x, y in micro_batches:
    loss = loss_fn(model(x),y)/n
    loss.backward()
    total_loss += loss.item()
  optimizer.step()
  return total_loss

In [39]:
model = nn.Linear(4,2)
opt = torch.optim.SGD(model.parameters(), lr=0.01)
loss = accumulation_step(model,opt, nn.MSELoss(),
    [(torch.randn(2,4), torch.randn(2,2)) for _ in range(4)])


In [40]:
loss

1.9895358085632324

##Layer Norm

$$\text{LayerNorm}(x) = \gamma \cdot \frac{x - \mu}{\sqrt{\sigma^2 + \epsilon}} + \beta$$

In [41]:
def layer_norm(x, gamma, beta, eps=1e-5):
  x_mean = x.mean(dim=-1, keepdim=True)
  denominator = torch.sqrt(x.var(dim=-1, unbiased=False, keepdim=True)+eps)

  return gamma * (x-x_mean)/denominator +beta

In [42]:
x = torch.randn(2,8)
gamma = torch.ones(8)
beta = torch.zeros(8)
out = layer_norm(x, gamma, beta)


In [43]:
out

tensor([[ 0.5905,  0.2855, -0.1023, -0.0407, -0.5995,  0.5142,  1.5097, -2.1574],
        [ 0.6725, -0.5418, -0.4648,  0.2114, -0.0035, -0.6043, -1.4182,  2.1486]])

## Linear regression

In [44]:
class LinearRegression:
  def closed_form(self, X:torch.Tensor, y:torch.Tensor):
    N,D = X.shape
    X_aug = torch.cat([X, torch.ones(N,1)], dim=1)
    theta = torch.linalg.lstsq(X_aug, y).solution
    w = theta[:D]
    b = theta[D]
    return w.detach(), b.detach()
  def gradient_descent(self, X:torch.Tensor, y:torch.Tensor, lr: float = 0.01, step: int = 1000):
    N,D = X.shape
    w = torch.zeros(D)
    b = torch.tensor(0.0)

    for _ in range(step):
      pred = X @ w + b
      error = pred - y
      grad_w = (2.0/N)*(X.T @ error)
      grad_b = (2.0/N)*error.sum()
      w = w - lr * grad_w
      b = b - lr * grad_b
    return w,b
  def nn_linear(self, X:torch.Tensor, y:torch.Tensor, lr :float = 0.001,steps:int = 1000):
    N, D = X.shape
    layer = nn.Linear(D,1)
    optimizer = torch.optim.SGD(layer.parameters(), lr=lr)
    loss_fn = nn.MSELoss()
    for _ in range(steps):
      optimizer.zero_grad()
      pred = layer(X).squeeze(-1)
      loss = loss_fn(pred,y)
      loss.backward()
      optimizer.step()

      w = layer.weight.data.squeeze()
      b = layer.bias.data
    return w,b



In [45]:
torch.manual_seed(42)
X = torch.randn(100,3)
true_w = torch.tensor([2.0, -1.0, 0.5])
y = X @ true_w + 3.0

model = LinearRegression()
for name, method in [("Closed-form", model.closed_form),
                      ("Grad Descent", lambda X, y: model.gradient_descent(X, y, lr=0.05, step=2000)),
                      ("nn.Linear", lambda X, y: model.nn_linear(X, y, lr=0.05, steps=2000))]:
    w, b = method(X, y)
    print(f"{name:13s}  w={w.tolist()}  b={b.item():.4f}")
print(f"{'True':13s}  w={true_w.tolist()}  b=3.0000")

Closed-form    w=[2.0, -1.0000001192092896, 0.5]  b=3.0000
Grad Descent   w=[2.0, -0.9999996423721313, 0.49999988079071045]  b=3.0000
nn.Linear      w=[2.0000009536743164, -0.9999997019767761, 0.4999997615814209]  b=3.0000
True           w=[2.0, -1.0, 0.5]  b=3.0000


##BatchNorm
$$\text{BN}(x) = \gamma \cdot \frac{x - \mu_B}{\sqrt{\sigma_B^2 + \epsilon}} + \beta$$

In [46]:
def batch_norm(x, gamma, beta, running_mean, running_var, eps=1e-5, momentum=0.1,training=True):
  if training:
    batch_mean = x.mean(dim=0)
    batch_var = x.var(dim=0, unbiased=False)

    running_mean.mul_(1 - momentum).add_(momentum * batch_mean.detach())
    running_var.mul_(1 - momentum).add_(momentum * batch_var.detach())

    mean = batch_mean
    var = batch_var
  else:
    mean = running_mean
    var = running_var
  x_norm = (x - mean) / torch.sqrt(var + eps)
  return gamma * x_norm + beta

In [47]:
x = torch.randn(8,4)
gamma = torch.ones(4)
beta = torch.zeros(4)

running_mean = torch.zeros(4)
running_var = torch.ones(4)

out_train = batch_norm(x, gamma, beta, running_mean, running_var, training = True)
out_val = batch_norm(x, gamma, beta, running_mean, running_var, training=False)

In [48]:
out_train

tensor([[-0.7767, -1.1030,  1.5462, -1.8361],
        [-0.6194,  0.3117,  0.2431,  0.3631],
        [-1.0508,  0.7455, -1.8702, -0.9611],
        [-0.7682,  0.1746, -0.4730,  0.3676],
        [ 1.4651,  0.0662,  1.2155,  0.1659],
        [-0.1158, -1.0933, -0.3505,  0.3208],
        [ 0.0912,  1.9486,  0.1446,  1.8047],
        [ 1.7745, -1.0503, -0.4556, -0.2249]])

In [49]:
out_val

tensor([[ 0.3938, -0.7457,  1.0943, -1.4744],
        [ 0.4272,  0.3658,  0.0159,  0.0995],
        [ 0.3357,  0.7065, -1.7330, -0.8482],
        [ 0.3956,  0.2581, -0.5767,  0.1027],
        [ 0.8691,  0.1729,  0.8206, -0.0417],
        [ 0.5339, -0.7381, -0.4754,  0.0693],
        [ 0.5778,  1.6518, -0.0656,  1.1312],
        [ 0.9347, -0.7043, -0.5624, -0.3213]])

## RMSNorm
$$\text{RMSNorm}(x) = \frac{x}{\text{RMS}(x)} \cdot w, \quad \text{RMS}(x) = \sqrt{\frac{1}{d}\sum x_i^2 + \epsilon}$$


In [50]:
def rms_norm(x, weight, eps=1e-6):
    rms = torch.sqrt(x.pow(2).mean(dim=-1, keepdim=True)+eps)
    return x / rms * weight

In [51]:
x = torch.randn(2,8)
out = rms_norm(x, torch.ones(8))


In [52]:
x

tensor([[-0.7939,  0.3752,  0.0879, -1.2415, -0.3203, -0.8444, -0.5513,  1.9890],
        [ 1.9003,  1.6951,  0.0281, -0.1754, -1.7735, -0.7046, -0.3947,  1.8868]])

In [53]:
out

tensor([[-0.8257,  0.3903,  0.0914, -1.2912, -0.3331, -0.8782, -0.5734,  2.0686],
        [ 1.4430,  1.2872,  0.0213, -0.1332, -1.3467, -0.5351, -0.2997,  1.4328]])

## SwiGLU MLP

$$\text{SwiGLU}(x) = \text{down\_proj}\big(\text{SiLU}(\text{gate\_proj}(x)) \odot \text{up\_proj}(x)\big)$$



In [54]:
import torch.nn as nn
import torch.nn.functional as F

In [55]:
class SwiGLUMLP(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.gate_proj = nn.Linear(d_model, d_ff)
        self.up_proj = nn.Linear(d_model, d_ff)
        self.down_proj = nn.Linear(d_ff, d_model)

    def forward(self, x):
        return self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x))

In [56]:
mlp = SwiGLUMLP(d_model=64, d_ff = 128)
x = torch.randn(2,8,64)
mlp(x).shape
print('Params:', sum(p.numel() for p in mlp.parameters()))

Params: 24896


## Conv2D

In [57]:
def my_conv2d(x, weight, bias=None, stride = 1, padding = 0):
  if padding > 0:
    x = F.pad(x, [padding] * 4)
  B, C_in, H, W = x.shape
  C_out, _, kH, kW = weight.shape
  H_out = (H - kH) // stride + 1
  W_out = (W - kW) // stride + 1
  patches = x.unfold(2, kH, stride).unfold(3, kW, stride)
  out = torch.einsum('bihwjk,oijk->bohw', patches, weight)
  if bias is not None:
    out = out + bias.view(1,-1,1,1)
  return out

In [58]:
x = torch.randn(1,3,8,8)
w = torch.randn(16,3,3,3)
my_conv2d(x,w).shape

torch.Size([1, 16, 6, 6])